In [1]:
# ============================================================
# CA6000 (Kaggle PS S5E12) — Data Cleaning & Preprocessing
# Output: X_train_proc, X_val_proc, y_train, y_val, X_test_proc
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin, clone
import joblib
import scipy.sparse as sp

SEED = 42
TARGET_COL = "diagnosed_diabetes"
ID_COL = "id"


In [2]:

# ----------------------------
# 1) Robust path resolver (Kaggle / Colab / local / /mnt/data)
# ----------------------------
from pathlib import Path

def resolve_dataset_paths(prefer_dir="/content"):
    candidates = [Path(prefer_dir), Path("/mnt/data"), Path(".")]

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.append(kaggle_input)

    def find_file(root: Path, filename: str):
        direct = root / filename
        if direct.exists():
            return direct
        hits = list(root.rglob(filename))
        return hits[0] if hits else None

    train_path = test_path = sub_path = None
    for root in candidates:
        tp = find_file(root, "train.csv")
        te = find_file(root, "test.csv")
        ss = find_file(root, "sample_submission.csv")
        if tp is not None and te is not None:
            train_path, test_path, sub_path = tp, te, ss
            break

    if train_path is None or test_path is None:
        raise FileNotFoundError("Cannot find train.csv/test.csv under preferred dirs.")

    return str(train_path), str(test_path), (str(sub_path) if sub_path else None)

TRAIN_PATH, TEST_PATH, SUB_PATH = resolve_dataset_paths("/content")
print(TRAIN_PATH, TEST_PATH, SUB_PATH)

/kaggle/input/playground-series-s5e12/train.csv /kaggle/input/playground-series-s5e12/test.csv /kaggle/input/playground-series-s5e12/sample_submission.csv


In [3]:
# ----------------------------
# 2) Load data
# ----------------------------
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("\nShapes:")
print("train:", train_df.shape)
print("test :", test_df.shape)
print("\nTrain head:")
display(train_df.head(3))


Shapes:
train: (700000, 26)
test : (300000, 25)

Train head:


,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0


In [4]:
# =========================
# 2.5 External ORIG features (Kaggle)
# Create: orig_mean_<col>, orig_count_<col>
# =========================

import numpy as np
import pandas as pd
from pathlib import Path

TARGET_COL = "diagnosed_diabetes"
ID_COL = "id"

# 1) find external csv under /kaggle/input (exclude competition folder)
def find_external_csv_excluding_competition():
    root = Path("/kaggle/input")
    if not root.exists():
        raise FileNotFoundError("/kaggle/input not found (not on Kaggle?)")

    all_csvs = list(root.rglob("*.csv"))
    if not all_csvs:
        raise FileNotFoundError("No csv under /kaggle/input. Did you add the dataset?")

    # exclude competition dataset
    def is_competition(p: Path) -> bool:
        s = str(p).lower()
        return "playground-series-s5e12" in s or "playground_series_s5e12" in s

    ext_csvs = [p for p in all_csvs if not is_competition(p)]
    if not ext_csvs:
        raise FileNotFoundError(
            "Found only competition csv under /kaggle/input. "
            "Please ensure you added the external dataset (Diabetes Health Indicators)."
        )

    # prefer filenames with health/indicator keyword
    prefer = [p for p in ext_csvs if ("health" in str(p).lower()) or ("indicator" in str(p).lower())]
    candidates = prefer if prefer else ext_csvs

    # choose largest file (usually main dataset)
    candidates = sorted(candidates, key=lambda p: p.stat().st_size, reverse=True)
    return str(candidates[0])

ORIG_PATH = find_external_csv_excluding_competition()
orig_df = pd.read_csv(ORIG_PATH)

print("✅ External ORIG csv:", ORIG_PATH)
print("orig_df shape:", orig_df.shape)

# 2) detect external target column
EXT_TARGET_CANDIDATES = ["Diabetes_binary", "diabetes_binary", "Diabetes_012", "diabetes_012", TARGET_COL]
ext_target = None
for c in EXT_TARGET_CANDIDATES:
    if c in orig_df.columns:
        ext_target = c
        break
if ext_target is None:
    raise ValueError(
        f"Cannot find external target column. Tried: {EXT_TARGET_CANDIDATES}. "
        f"Please check orig_df.columns."
    )

# convert external target to binary
orig_df[ext_target] = pd.to_numeric(orig_df[ext_target], errors="coerce")
if orig_df[ext_target].dropna().nunique() > 2:
    orig_df[ext_target] = (orig_df[ext_target] >= 1).astype(int)
else:
    orig_df[ext_target] = orig_df[ext_target].astype(int)

global_mean = float(orig_df[ext_target].mean())
print("✅ External target:", ext_target, "| external positive rate:", global_mean)

# 3) competition base feature columns
base_cols = [c for c in train_df.columns if c not in [ID_COL, TARGET_COL]]
common_cols = [c for c in base_cols if c in orig_df.columns]
skipped = [c for c in base_cols if c not in orig_df.columns]

print("Competition features:", len(base_cols))
print("Matched in external (exact name):", len(common_cols))
if skipped:
    print("⚠️ Skipped (not found in external by exact name):", skipped)

# 4) numeric binning helper
def quantile_edges(series: pd.Series, q=50):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return None
    bins = pd.qcut(s, q=q, duplicates="drop")
    cats = bins.cat.categories
    if len(cats) < 2:
        return None
    edges = np.r_[cats.left[0], cats.right.values]
    edges = np.unique(edges)
    return edges if len(edges) >= 3 else None

# 5) create external features (fix fillna TypeError by casting bins to object before map)
new_cols = []
NUM_Q = 50

for col in common_cols:
    is_numeric = pd.api.types.is_numeric_dtype(orig_df[col])

    if is_numeric:
        edges = quantile_edges(orig_df[col], q=NUM_Q)
        if edges is not None:
            ext_bins = pd.cut(pd.to_numeric(orig_df[col], errors="coerce"), bins=edges, include_lowest=True)
            mean_map = orig_df.groupby(ext_bins, observed=True)[ext_target].mean()
            count_map = orig_df.groupby(ext_bins, observed=True).size()

            tr_bins = pd.cut(pd.to_numeric(train_df[col], errors="coerce"), bins=edges, include_lowest=True)
            te_bins = pd.cut(pd.to_numeric(test_df[col],  errors="coerce"), bins=edges, include_lowest=True)

            mcol = f"orig_mean_{col}"
            ccol = f"orig_count_{col}"

            train_df[mcol] = tr_bins.astype("object").map(mean_map).fillna(global_mean).astype(np.float32)
            test_df[mcol]  = te_bins.astype("object").map(mean_map).fillna(global_mean).astype(np.float32)

            train_df[ccol] = tr_bins.astype("object").map(count_map).fillna(0).astype(np.float32)
            test_df[ccol]  = te_bins.astype("object").map(count_map).fillna(0).astype(np.float32)

            new_cols += [mcol, ccol]
        else:
            # fallback for low-cardinality numeric
            mean_map = orig_df.groupby(col, observed=True)[ext_target].mean()
            count_map = orig_df[col].value_counts(dropna=False)

            mcol = f"orig_mean_{col}"
            ccol = f"orig_count_{col}"

            train_df[mcol] = train_df[col].map(mean_map).fillna(global_mean).astype(np.float32)
            test_df[mcol]  = test_df[col].map(mean_map).fillna(global_mean).astype(np.float32)

            train_df[ccol] = train_df[col].map(count_map).fillna(0).astype(np.float32)
            test_df[ccol]  = test_df[col].map(count_map).fillna(0).astype(np.float32)

            new_cols += [mcol, ccol]
    else:
        mean_map = orig_df.groupby(col, observed=True)[ext_target].mean()
        count_map = orig_df[col].value_counts(dropna=False)

        mcol = f"orig_mean_{col}"
        ccol = f"orig_count_{col}"

        train_df[mcol] = train_df[col].map(mean_map).fillna(global_mean).astype(np.float32)
        test_df[mcol]  = test_df[col].map(mean_map).fillna(global_mean).astype(np.float32)

        train_df[ccol] = train_df[col].map(count_map).fillna(0).astype(np.float32)
        test_df[ccol]  = test_df[col].map(count_map).fillna(0).astype(np.float32)

        new_cols += [mcol, ccol]

print(f"✅ Added {len(new_cols)} external features.")
print("train_df shape now:", train_df.shape)
print("test_df  shape now:", test_df.shape)
print("Example new cols:", new_cols[:10])

print("ORIG_PATH =", ORIG_PATH)
print("orig_df.shape =", orig_df.shape)

✅ External ORIG csv: /kaggle/input/diabetes-health-indicators-dataset/diabetes_dataset.csv
orig_df shape: (100000, 31)
✅ External target: diagnosed_diabetes | external positive rate: 0.59998
Competition features: 24
Matched in external (exact name): 24
✅ Added 48 external features.
train_df shape now: (700000, 74)
test_df  shape now: (300000, 73)
Example new cols: ['orig_mean_age', 'orig_count_age', 'orig_mean_alcohol_consumption_per_week', 'orig_count_alcohol_consumption_per_week', 'orig_mean_physical_activity_minutes_per_week', 'orig_count_physical_activity_minutes_per_week', 'orig_mean_diet_score', 'orig_count_diet_score', 'orig_mean_sleep_hours_per_day', 'orig_count_sleep_hours_per_day']
ORIG_PATH = /kaggle/input/diabetes-health-indicators-dataset/diabetes_dataset.csv
orig_df.shape = (100000, 31)


In [5]:
# ----------------------------
# 3) Data audit & sanity checks (good for your report)
# ----------------------------
def basic_audit(train_df: pd.DataFrame, test_df: pd.DataFrame):
    # Required columns
    assert TARGET_COL in train_df.columns, f"Missing target '{TARGET_COL}' in train.csv"
    assert ID_COL in train_df.columns and ID_COL in test_df.columns, "Missing 'id' in train/test"

    # Column alignment (except target)
    train_features = [c for c in train_df.columns if c != TARGET_COL]
    assert set(train_features) == set(test_df.columns), "Train features != Test columns (schema mismatch)"

    # ID uniqueness
    assert train_df[ID_COL].is_unique, "Train id is not unique"
    assert test_df[ID_COL].is_unique, "Test id is not unique"

    # Duplicates
    dup_train = train_df.duplicated().sum()
    dup_test = test_df.duplicated().sum()

    # Missing summary
    miss_train = train_df.isnull().mean().sort_values(ascending=False)
    miss_test = test_df.isnull().mean().sort_values(ascending=False)

    # Target check
    y = train_df[TARGET_COL]
    unique_y = sorted(y.dropna().unique().tolist())

    print("\n[Audit] duplicates:", {"train": int(dup_train), "test": int(dup_test)})
    print("[Audit] top missing rate (train):")
    print(miss_train.head(10))
    print("[Audit] top missing rate (test):")
    print(miss_test.head(10))
    print("[Audit] target unique values:", unique_y)
    print("[Audit] target distribution:\n", y.value_counts(dropna=False))

    # ✅ NEW: external feature sanity check
    ext_cols = [c for c in train_df.columns if c.startswith("orig_mean_") or c.startswith("orig_count_")]
    print("\n[Audit] external feature count:", len(ext_cols))
    if len(ext_cols) == 0:
        print("⚠️ No external features detected.")
        print("   -> Make sure Cell 2.5 ran BEFORE this audit cell, and it added orig_* columns to BOTH train_df and test_df.")
    else:
        # ensure external cols exist in both train/test
        missing_in_test = [c for c in ext_cols if c not in test_df.columns]
        if missing_in_test:
            raise AssertionError(f"External cols missing in test_df (schema break): {missing_in_test[:10]}")

        # show quick stats (prove they are populated and numeric)
        print("[Audit] external missing rate (train avg):", float(train_df[ext_cols].isna().mean().mean()))
        print("[Audit] external missing rate (test  avg):", float(test_df[ext_cols].isna().mean().mean()))
        print("[Audit] external feature sample:", ext_cols[:6])
        print("[Audit] external feature stats (train, first 6):")
        print(train_df[ext_cols[:6]].describe().T[["mean", "std", "min", "max"]])

basic_audit(train_df, test_df)

# Convert target to int (0/1)
train_df[TARGET_COL] = train_df[TARGET_COL].astype(int)


[Audit] duplicates: {'train': 0, 'test': 0}
[Audit] top missing rate (train):
id                                    0.0
age                                   0.0
alcohol_consumption_per_week          0.0
physical_activity_minutes_per_week    0.0
diet_score                            0.0
sleep_hours_per_day                   0.0
screen_time_hours_per_day             0.0
bmi                                   0.0
waist_to_hip_ratio                    0.0
systolic_bp                           0.0
dtype: float64
[Audit] top missing rate (test):
id                                    0.0
age                                   0.0
alcohol_consumption_per_week          0.0
physical_activity_minutes_per_week    0.0
diet_score                            0.0
sleep_hours_per_day                   0.0
screen_time_hours_per_day             0.0
bmi                                   0.0
waist_to_hip_ratio                    0.0
systolic_bp                           0.0
dtype: float64
[Audit] target uni

In [6]:

# ----------------------------
# 4) Define column groups
# ----------------------------
# Categorical columns (object/string)
cat_cols = train_df.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [c for c in cat_cols if c != ID_COL]  # ensure id is not treated as category

# Binary columns (known 0/1 flags in this dataset)
bin_cols = ["family_history_diabetes", "hypertension_history", "cardiovascular_history"]
bin_cols = [c for c in bin_cols if c in train_df.columns]

# Numeric columns = all numeric excluding id/target/binary
num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in [ID_COL, TARGET_COL] + bin_cols]

print("Column groups:")
print("num_cols:", num_cols)
print("bin_cols:", bin_cols)
print("cat_cols:", cat_cols)


Column groups:
num_cols: ['age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'orig_mean_age', 'orig_count_age', 'orig_mean_alcohol_consumption_per_week', 'orig_count_alcohol_consumption_per_week', 'orig_mean_physical_activity_minutes_per_week', 'orig_count_physical_activity_minutes_per_week', 'orig_mean_diet_score', 'orig_count_diet_score', 'orig_mean_sleep_hours_per_day', 'orig_count_sleep_hours_per_day', 'orig_mean_screen_time_hours_per_day', 'orig_count_screen_time_hours_per_day', 'orig_mean_bmi', 'orig_count_bmi', 'orig_mean_waist_to_hip_ratio', 'orig_count_waist_to_hip_ratio', 'orig_mean_systolic_bp', 'orig_count_systolic_bp', 'orig_mean_diastolic_bp', 'orig_count_diastolic_bp', 'orig_mean_heart_rate', 'orig_count_heart_rate', 'orig_mean_chole

In [7]:
# Optional: verify binary columns truly contain only 0/1
for c in bin_cols:
    bad_vals = set(train_df[c].dropna().unique()) - {0, 1}
    if bad_vals:
        raise ValueError(f"Binary col '{c}' has unexpected values: {bad_vals}")

In [8]:
# ----------------------------
# 5) (Optional but nice) Range check for numeric columns
# ----------------------------
def numeric_range_report(df: pd.DataFrame, columns):
    desc = df[columns].describe(percentiles=[0.01, 0.5, 0.99]).T
    # Keep a compact view
    return desc[["min", "1%", "50%", "99%", "max", "mean", "std"]].sort_values("max", ascending=False)

range_report = numeric_range_report(train_df, num_cols)
print("\nNumeric range report (top 8 by max):")
display(range_report.head(8))



Numeric range report (top 8 by max):


,min,1%,50%,99%,max,mean,std
orig_count_cardiovascular_history,7920.0,7920.0,92080.0,92080.0,92080.0,89527.921875,14418.567383
orig_count_family_history_diabetes,21941.0,21941.0,78059.0,78059.0,78059.0,69674.890625,19956.292969
orig_count_hypertension_history,25080.0,25080.0,74920.0,74920.0,74920.0,65849.609375,19238.173828
orig_count_employment_status,6146.0,6146.0,60175.0,60175.0,60175.0,48978.468750,19028.378906
orig_count_smoking_status,20011.0,20011.0,59813.0,59813.0,59813.0,48149.683594,18073.042969
orig_count_gender,2013.0,47771.0,50216.0,50216.0,50216.0,48799.316406,3616.872314
orig_count_ethnicity,5049.0,5049.0,44997.0,44997.0,44997.0,32431.591797,14160.535156
orig_count_education_level,5100.0,5100.0,35037.0,44891.0,44891.0,36959.546875,10443.487305


In [9]:
# ----------------------------
# 6) Split data BEFORE fitting preprocessors (avoid leakage)
# ----------------------------
X = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL].values.astype(np.int32)

train_ids = X[ID_COL].values
test_ids = test_df[ID_COL].values

X = X.drop(columns=[ID_COL])
X_test = test_df.drop(columns=[ID_COL])

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("\nSplit shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape)
print("X_test :", X_test.shape)



Split shapes:
X_train: (560000, 72) y_train: (560000,)
X_val  : (140000, 72) y_val  : (140000,)
X_test : (300000, 72)


In [10]:
# ----------------------------
# 7) Custom transformer: quantile clipping for numeric outliers
#    (fit on training only)
# ----------------------------
class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower_q=0.005, upper_q=0.995):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lower_ = np.nanquantile(X, self.lower_q, axis=0)
        self.upper_ = np.nanquantile(X, self.upper_q, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.lower_, self.upper_)

In [11]:
# ----------------------------
# 8) Build preprocessing pipeline
#    - numeric: median impute -> clip -> standardize
#    - binary : most_frequent impute (keep 0/1)
#    - cate   : most_frequent impute -> Fold-safe target encoding (mean + count)
# ----------------------------

class MeanCountTargetEncoder(BaseEstimator, TransformerMixin):
    """Leakage-safe when fitted ONLY on training fold (we will do that in CV loop).
    Produces 2 features per categorical column: smoothed mean target + frequency(count).
    """
    def __init__(self, smoothing: float = 20.0):
        self.smoothing = smoothing

    def fit(self, X, y):
        X = pd.DataFrame(X).astype("object")
        y = pd.Series(y)

        self.global_mean_ = float(y.mean())
        self.mean_maps_ = []
        self.count_maps_ = []

        for j in range(X.shape[1]):
            col = X.iloc[:, j]
            stats = y.groupby(col).agg(["mean", "count"])
            smooth_mean = (stats["mean"] * stats["count"] + self.global_mean_ * self.smoothing) / (stats["count"] + self.smoothing)

            self.mean_maps_.append(smooth_mean)
            self.count_maps_.append(stats["count"])

        return self

    def transform(self, X):
        X = pd.DataFrame(X).astype("object")

        out_cols = []
        for j in range(X.shape[1]):
            col = X.iloc[:, j]
            m = col.map(self.mean_maps_[j]).fillna(self.global_mean_).astype(np.float32).to_numpy()
            c = col.map(self.count_maps_[j]).fillna(0).astype(np.float32).to_numpy()
            out_cols.append(m)
            out_cols.append(c)

        return np.vstack(out_cols).T  # (n_samples, 2 * n_cat_cols)


numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clipper", QuantileClipper(lower_q=0.005, upper_q=0.995)),
    ("scaler", StandardScaler())
])

binary_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("te", MeanCountTargetEncoder(smoothing=20.0))
])

# NOTE:
# - After switching OneHot -> TargetEncoding, output becomes dense (numpy array).
# - This is OK for XGBoost and also convenient for neural nets.
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("bin", binary_pipe, bin_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.0,  # force dense output
    verbose_feature_names_out=False
)

In [12]:
# ----------------------------
# 9) Fit on train, transform val/test
# IMPORTANT: pass y_train into fit_transform because categorical target encoder needs y
# ----------------------------
X_train_proc = preprocess.fit_transform(X_train, y_train)
X_val_proc = preprocess.transform(X_val)
X_test_proc = preprocess.transform(X_test)

# Cast to float32 for models (works for dense or sparse matrices)
X_train_proc = X_train_proc.astype(np.float32)
X_val_proc = X_val_proc.astype(np.float32)
X_test_proc = X_test_proc.astype(np.float32)

print("Processed shapes:")
print("X_train_proc:", X_train_proc.shape)
print("X_val_proc  :", X_val_proc.shape)
print("X_test_proc :", X_test_proc.shape)

# Safety checks (support sparse/dense)
def assert_no_nan(arr, name):
    if sp.issparse(arr):
        assert not np.isnan(arr.data).any(), f"NaNs remain in {name}"
    else:
        assert not np.isnan(arr).any(), f"NaNs remain in {name}"

assert_no_nan(X_train_proc, "X_train_proc")
assert_no_nan(X_val_proc, "X_val_proc")
assert_no_nan(X_test_proc, "X_test_proc")

Processed shapes:
X_train_proc: (560000, 78)
X_val_proc  : (140000, 78)
X_test_proc : (300000, 78)


In [13]:
# ----------------------------
# 10) Save artifacts for reproducibility
# ----------------------------
artifact = {
    "id_col": ID_COL,
    "target_col": TARGET_COL,
    "num_cols": num_cols,
    "bin_cols": bin_cols,
    "cat_cols": cat_cols,
    "preprocess": preprocess,
}

joblib.dump(artifact, "preprocess_artifact.joblib")
print("\nSaved preprocess artifact -> preprocess_artifact.joblib")

# Optional: save processed arrays (may be large, enable if you want)
# np.save("X_train_proc.npy", X_train_proc)
# np.save("X_val_proc.npy", X_val_proc)
# np.save("X_test_proc.npy", X_test_proc)
# np.save("y_train.npy", y_train)
# np.save("y_val.npy", y_val)

print("\n✅ Ready for model training stage:")
print("Use X_train_proc, y_train, X_val_proc, y_val, X_test_proc")


Saved preprocess artifact -> preprocess_artifact.joblib

✅ Ready for model training stage:
Use X_train_proc, y_train, X_val_proc, y_val, X_test_proc


In [14]:
# ----------------------------
# 11) 5-fold OOF XGBoost (AUC) + Test bagging
# NOTE: preprocess contains target encoder, so we MUST call fit_transform(X_tr, y_tr) inside each fold.
# ----------------------------
import os
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

X_full_raw = train_df.drop(columns=[TARGET_COL, ID_COL])
y_full = train_df[TARGET_COL].values.astype(np.int32)
X_test_raw = test_df.drop(columns=[ID_COL])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_pred = np.zeros(len(y_full), dtype=np.float32)
test_pred_bag = np.zeros(len(test_df), dtype=np.float32)

# ---- Safe GPU detection (compiled + runtime) ----
cuda_visible = os.environ.get("CUDA_VISIBLE_DEVICES", "")
use_cuda = bool(xgb.build_info().get("USE_CUDA", False)) and (cuda_visible not in ("", "-1"))
print(f"[XGBoost] USE_CUDA build={bool(xgb.build_info().get('USE_CUDA', False))} | CUDA_VISIBLE_DEVICES='{cuda_visible}' | use_cuda={use_cuda}")

# Base params (per-fold scale_pos_weight computed inside loop)
xgb_params = dict(
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
    random_state=SEED,
    early_stopping_rounds=200,  # xgboost>=2.0/3.x: in constructor
)

# add device only if usable
if use_cuda:
    xgb_params["device"] = "cuda"

fold_scores, best_iters = [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full_raw, y_full), start=1):
    X_tr_raw, X_val_raw = X_full_raw.iloc[tr_idx], X_full_raw.iloc[val_idx]
    y_tr, y_val_fold = y_full[tr_idx], y_full[val_idx]

    # 1) Fold-specific preprocessing (fit on train fold only) —— leakage safe
    pre_fold = clone(preprocess)
    X_tr_proc = pre_fold.fit_transform(X_tr_raw, y_tr).astype(np.float32)
    X_val_proc = pre_fold.transform(X_val_raw).astype(np.float32)
    X_test_proc = pre_fold.transform(X_test_raw).astype(np.float32)

    # 2) Fold-specific class weight
    pos = int((y_tr == 1).sum())
    neg = int((y_tr == 0).sum())
    scale_pos = neg / max(pos, 1)

    # 3) Train model
    model = xgb.XGBClassifier(**xgb_params, scale_pos_weight=scale_pos)

    model.fit(
        X_tr_proc, y_tr,
        eval_set=[(X_val_proc, y_val_fold)],
        verbose=50,
    )

    # 4) Predict using best_iteration
    best_iter = getattr(model, "best_iteration", None)
    if best_iter is not None:
        pred_val = model.predict_proba(X_val_proc, iteration_range=(0, best_iter + 1))[:, 1]
        pred_test = model.predict_proba(X_test_proc, iteration_range=(0, best_iter + 1))[:, 1]
        best_iters.append(int(best_iter))
    else:
        pred_val = model.predict_proba(X_val_proc)[:, 1]
        pred_test = model.predict_proba(X_test_proc)[:, 1]
        best_iters.append(0)

    oof_pred[val_idx] = pred_val.astype(np.float32)
    test_pred_bag += pred_test.astype(np.float32) / skf.n_splits

    fold_auc = roc_auc_score(y_val_fold, pred_val)
    fold_scores.append(float(fold_auc))
    print(f"Fold {fold}: AUC={fold_auc:.5f}, best_iter={best_iters[-1]}")

oof_auc = roc_auc_score(y_full, oof_pred)
valid_best_iters = [b for b in best_iters if b > 0]
avg_best_iter = int(np.mean(valid_best_iters)) if valid_best_iters else 500

print(f"OOF AUC: {oof_auc:.5f}")
print(f"Fold AUCs: {[round(s, 5) for s in fold_scores]}")
print(f"Avg best_iter (rounded): {avg_best_iter}")

# test_pred_bag is ready if you want to submit the 5-fold bagging result
print("test_pred_bag shape:", test_pred_bag.shape)

[XGBoost] USE_CUDA build=True | CUDA_VISIBLE_DEVICES='' | use_cuda=False
[0]	validation_0-auc:0.67418
[50]	validation_0-auc:0.69802
[100]	validation_0-auc:0.70458
[150]	validation_0-auc:0.70729
[200]	validation_0-auc:0.70954
[250]	validation_0-auc:0.71118
[300]	validation_0-auc:0.71245
[350]	validation_0-auc:0.71380
[400]	validation_0-auc:0.71534
[450]	validation_0-auc:0.71677
[500]	validation_0-auc:0.71825
[550]	validation_0-auc:0.71917
[600]	validation_0-auc:0.71964
[650]	validation_0-auc:0.72035
[700]	validation_0-auc:0.72090
[750]	validation_0-auc:0.72167
[800]	validation_0-auc:0.72218
[850]	validation_0-auc:0.72257
[900]	validation_0-auc:0.72294
[950]	validation_0-auc:0.72319
[1000]	validation_0-auc:0.72350
[1050]	validation_0-auc:0.72390
[1100]	validation_0-auc:0.72427
[1150]	validation_0-auc:0.72452
[1200]	validation_0-auc:0.72473
[1250]	validation_0-auc:0.72493
[1300]	validation_0-auc:0.72510
[1350]	validation_0-auc:0.72530
[1400]	validation_0-auc:0.72545
[1450]	validation_0-au

In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

SEED = 42
N_SPLITS = 5

y = train_df[TARGET_COL].astype(int).values
X_raw = train_df.drop(columns=[TARGET_COL])   # 还包含 id
test_raw = test_df.copy()

train_ids = X_raw[ID_COL].values
test_ids = test_raw[ID_COL].values

# drop id for modeling
X_raw = X_raw.drop(columns=[ID_COL])
test_raw = test_raw.drop(columns=[ID_COL])

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(X_raw, y))

print("Ready:", X_raw.shape, test_raw.shape, "pos rate =", y.mean())

Ready: (700000, 72) (300000, 72) pos rate = 0.6232957142857143


In [16]:
!pip -q install catboost

from catboost import CatBoostClassifier

oof_cat = np.zeros(len(X_raw), dtype=np.float32)
test_cat = np.zeros(len(test_raw), dtype=np.float32)

cat_params = dict(
    iterations=5000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    verbose=200,
    # task_type="GPU",  # 如果你 Kaggle 开了 GPU 可打开
)

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    X_tr_raw, X_va_raw = X_raw.iloc[tr_idx], X_raw.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    pre = clone(preprocess)
    X_tr = pre.fit_transform(X_tr_raw, y_tr).astype(np.float32)
    X_va = pre.transform(X_va_raw).astype(np.float32)
    X_te = pre.transform(test_raw).astype(np.float32)

    model = CatBoostClassifier(**cat_params)
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True, early_stopping_rounds=200)

    oof_cat[va_idx] = model.predict_proba(X_va)[:, 1].astype(np.float32)
    test_cat += model.predict_proba(X_te)[:, 1].astype(np.float32) / N_SPLITS

    auc = roc_auc_score(y_va, oof_cat[va_idx])
    print(f"[CatBoost] fold {fold} AUC = {auc:.5f}")

print("[CatBoost] OOF AUC =", roc_auc_score(y, oof_cat))

0:	test: 0.6783179	best: 0.6783179 (0)	total: 177ms	remaining: 14m 44s
200:	test: 0.7102160	best: 0.7102160 (200)	total: 20.2s	remaining: 8m 2s
400:	test: 0.7143727	best: 0.7143727 (400)	total: 39.9s	remaining: 7m 38s
600:	test: 0.7198777	best: 0.7198777 (600)	total: 59.6s	remaining: 7m 16s
800:	test: 0.7222257	best: 0.7222258 (799)	total: 1m 19s	remaining: 6m 55s
1000:	test: 0.7238694	best: 0.7238694 (1000)	total: 1m 39s	remaining: 6m 35s
1200:	test: 0.7250065	best: 0.7250065 (1200)	total: 1m 58s	remaining: 6m 15s
1400:	test: 0.7257962	best: 0.7258016 (1396)	total: 2m 18s	remaining: 5m 55s
1600:	test: 0.7263063	best: 0.7263087 (1598)	total: 2m 38s	remaining: 5m 36s
1800:	test: 0.7266337	best: 0.7266363 (1798)	total: 2m 58s	remaining: 5m 16s
2000:	test: 0.7268851	best: 0.7269010 (1977)	total: 3m 18s	remaining: 4m 57s
2200:	test: 0.7271572	best: 0.7271572 (2199)	total: 3m 37s	remaining: 4m 37s
2400:	test: 0.7273351	best: 0.7273486 (2383)	total: 3m 57s	remaining: 4m 17s
2600:	test: 0.727

In [18]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def assert_finite(name, X):
    if not np.isfinite(X).all():
        bad = np.where(~np.isfinite(X))
        print(name, "non-finite count:", len(bad[0]))
        raise ValueError(f"{name} contains NaN/Inf")

class NumpyDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i):
        if self.y is None: return self.X[i]
        return self.X[i], self.y[i]

class FTTransformer(nn.Module):
    def __init__(self, n_features, d_token=64, n_heads=8, n_layers=3, dropout=0.1):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_features, d_token) * 0.01)
        self.b = nn.Parameter(torch.zeros(n_features, d_token))
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))

        enc = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token*4,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, d_token),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_token, 1)
        )

    def forward(self, x):
        tokens = x.unsqueeze(-1) * self.W.unsqueeze(0) + self.b.unsqueeze(0)
        cls = self.cls.expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        z = self.encoder(tokens)
        logits = self.head(z[:, 0, :]).squeeze(-1)
        return logits

def train_ft_once(X_tr, y_tr, X_va, y_va, X_te, epochs=5, batch_size=4096, lr=5e-4):
    tr_loader = DataLoader(NumpyDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, num_workers=2)
    va_loader = DataLoader(NumpyDataset(X_va, y_va), batch_size=batch_size, shuffle=False, num_workers=2)
    te_loader = DataLoader(NumpyDataset(X_te, None), batch_size=batch_size, shuffle=False, num_workers=2)

    model = FTTransformer(n_features=X_tr.shape[1], d_token=64, n_heads=8, n_layers=3, dropout=0.1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss()

    best_auc, best_state = -1, None

    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # ⭐
            opt.step()

        model.eval()
        va_pred = []
        with torch.no_grad():
            for xb, yb in va_loader:
                xb = xb.to(device)
                p = torch.sigmoid(model(xb)).cpu().numpy()
                va_pred.append(p)
        va_pred = np.concatenate(va_pred)

        if not np.isfinite(va_pred).all():
            raise ValueError("va_pred contains NaN/Inf (training unstable). Try smaller lr or fewer layers.")

        auc = roc_auc_score(y_va, va_pred)
        print(f"[FT] epoch {ep} val AUC = {auc:.5f}")

        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    # final val pred
    va_pred = []
    with torch.no_grad():
        for xb, yb in va_loader:
            xb = xb.to(device)
            va_pred.append(torch.sigmoid(model(xb)).cpu().numpy())
    va_pred = np.concatenate(va_pred)

    # test pred
    te_pred = []
    with torch.no_grad():
        for xb in te_loader:
            xb = xb.to(device)
            te_pred.append(torch.sigmoid(model(xb)).cpu().numpy())
    te_pred = np.concatenate(te_pred)

    return va_pred.astype(np.float32), te_pred.astype(np.float32), best_auc

# 5-fold
oof_ft = np.zeros(len(X_full_raw), dtype=np.float32)
test_ft = np.zeros(len(X_test_raw), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    X_tr_raw = X_full_raw.iloc[tr_idx]
    X_va_raw = X_full_raw.iloc[va_idx]
    y_tr = y_full[tr_idx]
    y_va = y_full[va_idx]

    pre = clone(preprocess)
    X_tr = pre.fit_transform(X_tr_raw, y_tr).astype(np.float32)
    X_va = pre.transform(X_va_raw).astype(np.float32)
    X_te = pre.transform(X_test_raw).astype(np.float32)

    assert_finite("X_tr", X_tr)
    assert_finite("X_va", X_va)
    assert_finite("X_te", X_te)

    va_pred, te_pred, best_auc = train_ft_once(
        X_tr, y_tr, X_va, y_va, X_te,
        epochs=5, batch_size=4096, lr=5e-4
    )

    oof_ft[va_idx] = va_pred
    test_ft += te_pred / N_SPLITS
    print(f"[FT] fold {fold} best AUC = {best_auc:.5f}")

print("[FT] OOF AUC =", roc_auc_score(y_full, oof_ft))


device: cuda
[FT] epoch 1 val AUC = 0.69479
[FT] epoch 2 val AUC = 0.69611
[FT] epoch 3 val AUC = 0.69455
[FT] epoch 4 val AUC = 0.69412
[FT] epoch 5 val AUC = 0.69836
[FT] fold 1 best AUC = 0.69836
[FT] epoch 1 val AUC = 0.69296
[FT] epoch 2 val AUC = 0.69205
[FT] epoch 3 val AUC = 0.69533
[FT] epoch 4 val AUC = 0.69620
[FT] epoch 5 val AUC = 0.69558
[FT] fold 2 best AUC = 0.69620
[FT] epoch 1 val AUC = 0.69278
[FT] epoch 2 val AUC = 0.69469
[FT] epoch 3 val AUC = 0.69397
[FT] epoch 4 val AUC = 0.69296
[FT] epoch 5 val AUC = 0.69465
[FT] fold 3 best AUC = 0.69469
[FT] epoch 1 val AUC = 0.69398
[FT] epoch 2 val AUC = 0.69534
[FT] epoch 3 val AUC = 0.69521
[FT] epoch 4 val AUC = 0.69587
[FT] epoch 5 val AUC = 0.69420
[FT] fold 4 best AUC = 0.69587
[FT] epoch 1 val AUC = 0.69501
[FT] epoch 2 val AUC = 0.69774
[FT] epoch 3 val AUC = 0.69583
[FT] epoch 4 val AUC = 0.69612
[FT] epoch 5 val AUC = 0.69833
[FT] fold 5 best AUC = 0.69833
[FT] OOF AUC = 0.6947761966113049


In [19]:
from sklearn.linear_model import LogisticRegression

oof_xgb = oof_pred
test_xgb = test_pred_bag

S_train = np.vstack([oof_xgb, oof_cat, oof_ft]).T
S_test  = np.vstack([test_xgb, test_cat, test_ft]).T

meta = LogisticRegression(max_iter=2000, random_state=SEED)
meta.fit(S_train, y)

stack_oof = meta.predict_proba(S_train)[:, 1]
print("[Stack] OOF AUC =", roc_auc_score(y, stack_oof))

stack_test = meta.predict_proba(S_test)[:, 1].astype(np.float32)

submission = pd.DataFrame({ID_COL: test_ids, TARGET_COL: stack_test})
submission.to_csv("submission_stacking.csv", index=False)
display(submission.head())
print("Saved submission_stacking.csv", submission.shape)

[Stack] OOF AUC = 0.7281947765030578


,id,diagnosed_diabetes
0,700000,0.443270
1,700001,0.741170
2,700002,0.800968
3,700003,0.377351
4,700004,0.905502


Saved submission_stacking.csv (300000, 2)


In [23]:
from sklearn.metrics import roc_auc_score

print("XGB OOF AUC =", roc_auc_score(y, oof_pred))
print("CAT OOF AUC =", roc_auc_score(y, oof_cat))
print("FT  OOF AUC =", roc_auc_score(y, oof_ft))

avg_oof = (oof_pred + oof_cat + oof_ft) / 3
print("Avg OOF AUC =", roc_auc_score(y, avg_oof))  # 算术平均对比

print("Stack OOF AUC =", roc_auc_score(y, stack_oof))

XGB OOF AUC = 0.7275595529338112
CAT OOF AUC = 0.7277348757702988
FT  OOF AUC = 0.6947761966113049
Avg OOF AUC = 0.7239800148004756
Stack OOF AUC = 0.7281947765030578


In [20]:
# ----------------------------
# 12) Final train on full data + generate submission
# IMPORTANT: preprocess contains target encoder, so we MUST call fit_transform(X_full, y_full).
# ----------------------------
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.base import clone

# Ensure raw matrices exist even if previous cell wasn't run
X_full_raw = train_df.drop(columns=[TARGET_COL, ID_COL])
y_full = train_df[TARGET_COL].values.astype(np.int32)
X_test_raw = test_df.drop(columns=[ID_COL])

if 'test_ids' not in locals():
    test_ids = test_df[ID_COL].values

# Recompute class weight and xgb_params if missing
scale_pos_final = (y_full == 0).sum() / (y_full == 1).sum()

if 'xgb_params' not in locals():
    xgb_params = dict(
        n_estimators=5000,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        n_jobs=-1,
        random_state=SEED,
    )
else:
    xgb_params = dict(xgb_params)
    xgb_params['scale_pos_weight'] = scale_pos_final

# Use avg_best_iter from OOF cell if available
final_n_estimators = int(avg_best_iter) if 'avg_best_iter' in locals() else int(xgb_params.get("n_estimators", 5000))
xgb_params_final = dict(xgb_params)
xgb_params_final["n_estimators"] = final_n_estimators
xgb_params_final.pop("early_stopping_rounds", None)
xgb_params_final.pop("device", None)  # keep CPU-safe; you can re-add if you enabled GPU

print("[Final] n_estimators =", final_n_estimators)

# Fit preprocessing on full train (need y_full) then transform test
preprocess_final = clone(preprocess)
X_full_proc = preprocess_final.fit_transform(X_full_raw, y_full).astype(np.float32)
X_test_proc_final = preprocess_final.transform(X_test_raw).astype(np.float32)

# Ensure scale_pos_weight appears only once
xgb_params_final = dict(xgb_params_final)
xgb_params_final["scale_pos_weight"] = scale_pos_final  # keep it in dict

model_final = xgb.XGBClassifier(**xgb_params_final)
model_final.fit(X_full_proc, y_full, verbose=50)

test_pred = model_final.predict_proba(X_test_proc_final)[:, 1]
submission = pd.DataFrame({ID_COL: test_ids, TARGET_COL: test_pred})
submission.to_csv("submission.csv", index=False)

print("Saved submission.csv with shape", submission.shape)
display(submission.head())

[Final] n_estimators = 3873
Saved submission.csv with shape (300000, 2)


,id,diagnosed_diabetes
0,700000,0.339046
1,700001,0.612486
2,700002,0.651224
3,700003,0.289542
4,700004,0.889654


In [21]:
print(submission.isna().sum())
print(submission[TARGET_COL].min(), submission[TARGET_COL].max())
print(submission[ID_COL].nunique(), submission.shape[0])

id                    0
diagnosed_diabetes    0
dtype: int64
0.021148044615983963 0.9916580319404602
300000 300000


In [22]:
assert submission.shape[0] == 300000
assert list(submission.columns) == ["id", "diagnosed_diabetes"]
assert submission["diagnosed_diabetes"].between(0, 1).all()
assert submission["id"].is_unique
